# Case Study: Selecting a Drug for Geographic Analysis

**Prerequisite:** Run `00_build_database.ipynb` first.

This notebook demonstrates the drug selection methodology used to identify candidates for geographic prescribing analysis. Lenalidomide (Revlimid) is used as the worked example — it was the highest-cost drug in Medicare Part D in 2023 at $3.86 billion nationally.

The same methodology applies to any drug in the dataset and is what powers the Streamlit app (`app.py`).

**Why lenalidomide?**
Lenalidomide is an oral immunomodulatory agent used almost exclusively for multiple myeloma. Because it is single-indication, its prescribing geography is a reasonable proxy for regional multiple myeloma treatment access — making it analytically meaningful for disease burden surveillance.

In [ ]:
import sqlite3
import pandas as pd

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"
conn = sqlite3.connect(db_path)

## Step 1: National Summary

Confirm national totals and identify all name variants in the dataset. Part D data sometimes splits brand and generic names into separate rows — these are aggregated in the map.

In [ ]:
national = pd.read_sql_query("""
    SELECT Brnd_Name, Gnrc_Name, Tot_Prscrbrs, Tot_Clms, Tot_Benes,
           ROUND(Tot_Drug_Cst, 0) AS Tot_Drug_Cst,
           ROUND(Tot_Drug_Cst / Tot_Benes, 0) AS Cost_Per_Bene
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'National'
      AND LOWER(Gnrc_Name) LIKE '%lenalidomide%'
    ORDER BY Tot_Clms DESC
""", conn)

national

## Step 2: State Coverage Check

A drug needs near-complete state coverage for a choropleth map to be meaningful. Brand and generic rows are aggregated per state using SUM so each state produces one row.

The enrollment table is joined here to also verify that the FIPS join key aligns correctly before building the normalized metric.

In [ ]:
states = pd.read_sql_query("""
    SELECT p.Prscrbr_Geo_Desc AS state,
           e.BENE_STATE_ABRVTN AS state_abbr,
           SUM(p.Tot_Prscrbrs) AS tot_prscrbrs,
           SUM(p.Tot_Clms) AS tot_clms,
           SUM(p.Tot_Benes) AS tot_benes,
           ROUND(SUM(p.Tot_Drug_Cst), 0) AS tot_cost,
           e.PRSCRPTN_DRUG_TOT_BENES AS part_d_enrollees
    FROM part_d_geo p
    JOIN state_enrollment e ON p.Prscrbr_Geo_Cd = e.BENE_FIPS_CD
    WHERE p.Prscrbr_Geo_Lvl = 'State'
      AND LOWER(p.Gnrc_Name) LIKE '%lenalidomide%'
    GROUP BY p.Prscrbr_Geo_Desc, e.BENE_STATE_ABRVTN, e.PRSCRPTN_DRUG_TOT_BENES
    ORDER BY tot_clms DESC
""", conn)

print(f"States with data: {len(states)}")
states

## Step 3: Per-Capita Normalization

Raw claim counts reflect population size — larger states will always appear highest. Normalizing by Part D enrollment produces claims per 100,000 enrollees, which enables meaningful geographic comparison.

This is the same calculation used in the Streamlit app for every drug in the dataset.

In [ ]:
states['clms_per_100k'] = (states['tot_clms'] * 100000 / states['part_d_enrollees']).round(1)
states['cost_per_enrollee'] = (states['tot_cost'] / states['part_d_enrollees']).round(2)

print("Top 10 states by claims per 100k Part D enrollees:")
states[['state', 'tot_clms', 'clms_per_100k', 'cost_per_enrollee']].head(10)